In [1]:
import requests
import time
import json

# ---------------------------
# CONFIG
# ---------------------------
HEADERS = {
    "User-Agent": "academic-research-script/1.0"
}

REQUEST_DELAY = 1.2
POSTS_PER_SUBREDDIT = 6   # balanced sampling

# ---------------------------
# TIME PERIODS (UTC)
# ---------------------------
PERIODS = {
    "during_covid": (1583020800, 1625097600),
    "post_covid":   (1625097600, 1672444800),
    "ai_era":       (1704067200, 1735689600),
}

# ---------------------------
# SUBREDDITS
# ---------------------------
SUBREDDITS = [
    "developersIndia",
    "cscareerquestions",
    "ExperiencedDevs",
    "ITCareerQuestions",
    "programming",
    "learnprogramming",
    "careerguidance",
    "jobs",
    "recruitinghell",
    "antiwork",
    "technology",
    "artificial",
    "MachineLearning",
    "datascience",
]

# ---------------------------
# POST-FILTER KEYWORDS (NOT USED DURING COLLECTION)
# ---------------------------
BIAS_KEYWORDS = [
    "stress", "stressed", "burnout", "burned out", "burnt out",
    "anxiety", "anxious", "depression", "depressed",
    "overworked", "exhausted", "tired",
    "toxic", "toxic culture",
    "layoff", "laid off", "fired",
    "job insecurity", "career fear",
    "ai replace", "ai replacing", "automation",
    "pressure", "work pressure", "mental health",
]

BOT_PHRASES = [
    "i am a bot",
    "performed automatically",
    "contact the moderators",
    "this action was performed",
]

# ---------------------------
# HELPERS
# ---------------------------
def is_bot_comment(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in BOT_PHRASES)

def contains_bias_keywords(text: str) -> bool:
    text = text.lower()
    return any(k in text for k in BIAS_KEYWORDS)

def fetch_posts(subreddit, after, before, size=100):
    url = "https://api.pullpush.io/reddit/search/submission/"
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": size,
        "sort": "desc",
        "sort_type": "created_utc",
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.json().get("data", [])

def fetch_comments(post_id):
    url = "https://api.pullpush.io/reddit/comment/search/"
    params = {
        "link_id": post_id,
        "size": 10,
        "sort": "asc"
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=20)
    r.raise_for_status()

    comments = []
    for c in r.json().get("data", []):
        body = c.get("body", "").strip()
        if body and not is_bot_comment(body):
            comments.append(body)
        if len(comments) == 2:
            break

    return comments

# ---------------------------
# DATA COLLECTION (NEUTRAL)
# ---------------------------
all_docs = []
doc_id = 0

for period, (after, before) in PERIODS.items():
    print(f"\nCollecting period: {period}")

    for subreddit in SUBREDDITS:
        print(f"  r/{subreddit}")
        collected = 0
        posts = fetch_posts(subreddit, after, before)

        for post in posts:
            if collected >= POSTS_PER_SUBREDDIT:
                break

            title = post.get("title", "")
            body = post.get("selftext", "")

            if body.lower() in ["[deleted]", "[removed]"]:
                continue

            post_text = (title + " " + body).strip()
            if len(post_text) < 150:
                continue

            comments = fetch_comments(post["id"])
            if len(comments) < 1:
                continue

            record = {
                "doc_id": doc_id,
                "period": period,
                "subreddit": subreddit,
                "post_text": post_text,
                "comment_1": comments[0],
                "comment_2": comments[1] if len(comments) > 1 else ""
            }

            all_docs.append(record)
            doc_id += 1
            collected += 1
            time.sleep(REQUEST_DELAY)

# # ---------------------------
# # SAVE FULL NEUTRAL DATASET
# # ---------------------------
# with open("reddit_all_docs.json", "w", encoding="utf-8") as f:
#     json.dump(all_docs, f, indent=2, ensure_ascii=False)

# ---------------------------
# POST-COLLECTION FILTER (STRESS ONLY)
# ---------------------------
stress_docs = []

for doc in all_docs:
    combined = (
        doc["post_text"] + " " +
        doc["comment_1"] + " " +
        doc["comment_2"]
    )
    if contains_bias_keywords(combined):
        stress_docs.append(doc)

with open("reddit_stress_only.json", "w", encoding="utf-8") as f:
    json.dump(stress_docs, f, indent=2, ensure_ascii=False)

print("\nDone.")
print(f"Total neutral docs: {len(all_docs)}")
print(f"Stress-related docs: {len(stress_docs)}")
print("Files created:")
print(" - reddit_all_docs.json")
print(" - reddit_stress_only.json")



  r/developersIndia
  r/cscareerquestions
  r/ExperiencedDevs
  r/ITCareerQuestions
  r/programming
  r/learnprogramming
  r/careerguidance
  r/jobs
  r/recruitinghell
  r/antiwork
  r/technology
  r/artificial
  r/MachineLearning
  r/datascience

  r/developersIndia
  r/cscareerquestions
  r/ExperiencedDevs
  r/ITCareerQuestions
  r/programming
  r/learnprogramming
  r/careerguidance
  r/jobs
  r/recruitinghell
  r/antiwork
  r/technology
  r/artificial
  r/MachineLearning
  r/datascience

  r/developersIndia
  r/cscareerquestions
  r/ExperiencedDevs
  r/ITCareerQuestions
  r/programming
  r/learnprogramming
  r/careerguidance
  r/jobs
  r/recruitinghell
  r/antiwork
  r/technology
  r/artificial
  r/MachineLearning
  r/datascience

Done.
Total neutral docs: 236
Stress-related docs: 29
Files created:
 - reddit_all_docs.json
 - reddit_stress_only.json
